In [4]:
!powershell -Command "Invoke-WebRequest -Uri 'http://datasrc.tipdm.net:81/python/case/O2O/train.csv' -OutFile 'train.csv'"
!powershell -Command "Invoke-WebRequest -Uri 'http://datasrc.tipdm.net:81/python/case/O2O/test.csv' -OutFile 'test.csv'"
!powershell -Command "Invoke-WebRequest -Uri 'http://datasrc.tipdm.net:81/python/case/O2O/feature_name1.py' -OutFile 'feature_name1.py'"
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

data_train = pd.read_csv(r"F:\my item\O2O优惠券个性化投放\【下载】项目数据文件\train.csv")
data_test = pd.read_csv(r"F:\my item\O2O优惠券个性化投放\【下载】项目数据文件\test.csv")

# 预处理结果表
cleanfile_train = './tmp/clean_train.csv'
cleanfile_test = './tmp/clean_test.csv'

# 训练样本和测试样本进行合并，方便数据清洗
data = pd.concat([data_train, data_test], axis=0, join='outer')

# 对整个dataframe的空值进行处理
# 原数据中缺失值为"null"字符串,设置为 numpy.nan
data.iloc[:, :5] = data.iloc[:, :5].applymap(
    lambda x: np.nan if x == 'null' else x)

# 对日期类型的空值设置为 None (方便后面整列转成时间类型)
data.iloc[:, 5:] = data.iloc[:, 5:].applymap(
    lambda x: None if x == 'null' else x)

# 处理 date_received 字段
data['date_received'] = data['date_received'].astype('str').apply(
    lambda x: x.split('.')[0])

# 转为 datetime 格式
data['date_received'] = pd.to_datetime(data['date_received'])

# 处理 date 字段
data['date'] = data['date'].astype('str').apply(lambda x: x.split('.')[0])
data['date'] = pd.to_datetime(data['date'])

# 满减优惠改写成折扣率形式
data['discount_rate'] = data['discount_rate'].fillna('null')


def discount(x):
    if ':' in x:
        split = x.split(':')
        discount_rate = (int(split[0])-int(split[1]))/int(split[0])
        return round(discount_rate, 2)
    elif x == 'null':
        return np.nan
    else:
        return float(x)

data['discount_rate'] = data['discount_rate'].map(discount)

# 根据领券月份，提取清洗后的训练样本和测试样本
received_month = data['date_received'].apply(lambda x: x.month)
received_month.value_counts()
clean_train = data.loc[received_month != 7, :]  # 提取清洗后训练样本
clean_test = data.loc[received_month == 7, :]  # 提取清洗后测试样本
clean_test.drop('date', axis=1, inplace=True)  # 删除 date 列

# 导出数据
clean_train.to_csv(cleanfile_train, index=False)
clean_test.to_csv(cleanfile_test, index=False)


# 代码4-2 构建指标

# 用户、商户、优惠券的特征结果表
userfile = './tmp/data_user.csv'
merchantfile = './tmp/data_merchant.csv'
couponfile = './tmp/data_coupon.csv'

train_quality = clean_train.copy()
test_quality = clean_test.copy()

# 导入自定义用户、商户、优惠券的特征包
from feature_name1 import feature_name
data_user, data_merchant, data_coupon = feature_name(train_quality=
                                                     train_quality)

data_user.to_csv(userfile, index=False)  # 导出 data_user 表
data_merchant.to_csv(merchantfile, index=False)  # 导出 data_merchant数据表
data_coupon.to_csv(couponfile, index=False)  # 导出 data_coupon 数据表

# 对训练样本与特征类型表进行拼接
train_merge = pd.merge(data_user, train_quality, on='user_id')
train_merge = pd.merge(train_merge, data_merchant, on='merchant_id')
train_merge = pd.merge(train_merge, data_coupon, on='coupon_id', how='left')
train_merge.isnull().sum()  # 统计缺失值
train_merge.iloc[:, -2:] = train_merge.iloc[:, -2:].fillna(0)  # 缺失值填充
print('构建特征后训练样本的形状：', train_merge.shape)
trainfile = './tmp/train_cleaned.csv'  # 导出数据
train_merge.to_csv(trainfile, index=False)


# 对测试样本与特征类型表进行拼接
test_merge = pd.merge(data_user, test_quality, on='user_id')
test_merge = pd.merge(test_merge, data_merchant, on='merchant_id')
test_merge = pd.merge(test_merge, data_coupon, on='coupon_id', how='left')
test_merge.isnull().sum()  # 统计缺失值
test_merge.iloc[:, -2:] = test_merge.iloc[:, -2:].fillna(0)  # 缺失值填充

print('构建特征后测试样本的形状：', test_merge.shape)
testfile = './tmp/test_cleaned.csv'  # 导出数据
test_merge.to_csv(testfile, index=False)


# 代码 4-3 构建训练样本分类标签

# 建立训练样本分类标签
train_merge['class'] = 0  # 标签 0
train_merge.loc[(train_merge['date']-
                 train_merge['date_received']).dt.days<=15, 'class']=1  # 标签 1


# 删除非正负样本的数据（用户未领券的记录）
print(train_merge.shape)
train_merge = train_merge[train_merge['coupon_id'].notnull()]
print(train_merge.shape)
trainfile_class = './tmp/train_class.csv'  # 导出数据
train_merge.to_csv(trainfile_class, index=False)

构建特征后训练样本的形状： (1648881, 19)
构建特征后测试样本的形状： (100669, 18)
(1648881, 20)
(947279, 20)
